<a href="https://colab.research.google.com/github/candeperles/PDI-TP/blob/main/notebooks/17_modelo_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 17. Elección del modelo final

Va **después de la validación cruzada de Jamaica (16) y antes del test (18)**. Elige un modelo por familia,
una CNN y un detector usado como clasificador, junto con su umbral, y deja escrito por qué. Así el 18 se corre
**una sola vez** con la elección ya fijada (issue #45).

Usa solo lo que ya está en Drive:

1. **Val (162 imágenes):** las corridas vigentes del 15. De acá salieron los finalistas del #43.
2. **Validación cruzada de Jamaica (172 imágenes, fuera de fold):** las predicciones del 16, que decide entre
   los finalistas.

**El test no se carga ni se mira.** No entrena ni escribe nada: abre Drive en solo lectura y corre sin GPU.

**Limitación que hay que decir en el informe:** el 6/11 se corrió una versión preliminar del test con todas
las corridas vigentes. La regla de abajo es la que el 16 fijó antes de ver la CV, más una excepción para los
finalistas que no llegaron a completar los cinco folds. Esa excepción se agregó el 9/11, cuando el test
preliminar ya se había visto. Los modelos se entrenaron solo con train (738 imágenes): por falta de GPU no
se reentrenan con train + val.

In [ ]:
# --- Setup: correr siempre primero ---
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    repo = Path("/content/PDI-TP")
    if repo.is_dir():
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/candeperles/PDI-TP.git", str(repo)], check=True)
else:
    # Funciona al abrir Jupyter desde la raíz o desde notebooks/.
    repo = next((p for p in (Path.cwd(), Path.cwd().parent)
                 if (p / "pdi_tp" / "entorno.py").is_file()), None)
    if repo is None:
        raise FileNotFoundError("Abrí Jupyter desde la raíz del repositorio o desde notebooks/.")
os.chdir(repo)
sys.path.insert(0, str(repo))

import json

import pandas as pd
from IPython.display import display

from pdi_tp import comparacion, cv_jamaica, entorno, resultados, split

rutas = entorno.preparar(solo_lectura=True)  # monta Drive, pero no deja crear ni guardar corridas

## La regla

Está fijada antes de mirar las tablas de este notebook y repite la del 16:

1. **Finalistas:** los del #43 (`cv_jamaica.FINALISTAS`), elegidos en val. MobileNetV2 es opcional.
2. **Por familia, entre los finalistas que tienen los cinco folds**, se elige el de mayor **balanced accuracy**
   en la CV, con el umbral actual (0.5 en las CNN y 0.25 en los YOLO).
3. Si hay dos candidatos, se mide la diferencia pareada. **Si su intervalo incluye el 0**, gana el de menor
   costo medido: la mediana de segundos de entrenamiento por fold.
4. **Umbral:** cambia solo si el umbral elegido por grupos mejora claramente al actual (`mejora_clara` en
   `cv_jamaica.tabla_umbrales`). Si no, queda el actual.
5. **Épocas:** se registra la mediana de las mejores épocas de los folds. Sirve solo si alguna vez se
   reentrena con train + val (#45).

**Excepción (agregada el 9/11):** el 16 pedía no declarar ganador si falta un candidato. Si a un finalista
obligatorio le faltan folds, acá se elige igual al único completo y la tabla lo marca como *sin rival en la
CV*.

In [ ]:
FAMILIAS = {"clasificador": "CNN", "detector": "Detector usado como clasificador"}
FINALISTAS = [f"{modelo}/{modo}" for modelo, modo in cv_jamaica.FINALISTAS.items()]
OPCIONALES = {"mobilenetv2/imagenet_finetuning"}  # el #43 lo dejó opcional: si falta, no cuenta como rival
METRICA = "balanced_accuracy"
TITULOS = {"f1_macro": "F1 macro", "balanced_accuracy": "Balanced accuracy",
           "recall_sin_paneles": "Recall sin paneles", "roc_auc": "AUC"}


def familia_de(corrida):
    return comparacion.familia(corrida.split("/")[0])


def legible(tabla, extra=()):
    """Cada métrica como 'valor [inf, sup]', con las columnas extra al principio."""
    salida = tabla[["corrida", *extra]].copy()
    for metrica, titulo in TITULOS.items():
        salida[titulo] = [("—" if pd.isna(v) else f"{v:.3f} [{i:.3f}, {s:.3f}]")
                          for v, i, s in zip(tabla[metrica], tabla[f"{metrica}_inf"], tabla[f"{metrica}_sup"])]
    return salida.set_index("corrida")


print("Finalistas del #43:", FINALISTAS)

## 1. Val: de dónde salen los finalistas

Las mismas cuentas que el 15, para tener la elección completa en un solo lugar: todas las corridas vigentes de
cada familia, ordenadas por F1 macro en val, con intervalos de 95 % por bootstrap. Val también eligió los
checkpoints, así que estos números son optimistas. El mAP50 de los detectores es de Ultralytics, sin intervalo.

In [ ]:
corridas = comparacion.corridas_vigentes(resultados.cargar_corridas(rutas))
corridas = corridas[corridas["vigente"] & corridas["familia"].isin(FAMILIAS)]
VIGENTES = corridas["corrida"].tolist()

val = comparacion.intervalos(resultados.cargar_predicciones(rutas), VIGENTES)
val["familia"] = val["corrida"].map(familia_de).map(FAMILIAS)
val["finalista"] = val["corrida"].isin(FINALISTAS).map({True: "sí", False: ""})
mapas = comparacion.tabla_deteccion(rutas.resultados_equipo, VIGENTES)
mapas = mapas[mapas["conjunto"] == "val"].set_index("corrida")["map50"]
val["mAP50"] = val["corrida"].map(mapas).map(lambda v: "" if pd.isna(v) else f"{v:.3f}")
val = val.sort_values(["familia", "f1_macro"], ascending=[True, False])

faltan_val = [c for c in FINALISTAS if c not in VIGENTES]
print("Finalistas sin corrida vigente en val:", faltan_val or "ninguno")
legible(val, extra=("familia", "finalista", "mAP50"))

## 2. Validación cruzada de Jamaica

Se cargan las predicciones fuera de fold del 16. `cv_jamaica.cargar_predicciones` solo junta modelos con los
cinco folds íntegros y de la misma receta: los incompletos no entran.

In [ ]:
folds = split.folds_jamaica(rutas, n=5, semilla=42)
cv, estado = cv_jamaica.cargar_predicciones(rutas, folds)
estado["corrida"] = estado["modelo"] + "/" + estado["modo"]
folds_completos = estado.groupby("corrida")["completa"].sum().astype(int).reindex(FINALISTAS, fill_value=0)
CON_CV = [c for c in FINALISTAS if folds_completos[c] == 5]
display(pd.DataFrame({"folds completos": folds_completos,
                      "entra": folds_completos.index.isin(CON_CV),
                      "opcional": folds_completos.index.isin(OPCIONALES)}))
if not CON_CV:
    raise RuntimeError("Ningún finalista tiene los cinco folds: no hay con qué decidir.")

resumen_cv = comparacion.intervalos(cv, CON_CV)
resumen_cv["familia"] = resumen_cv["corrida"].map(familia_de).map(FAMILIAS)
print(f"CV: {resumen_cv['n'].iloc[0]} imágenes de Jamaica, {resumen_cv['n_sin_paneles'].iloc[0]} sin paneles.")
legible(resumen_cv.sort_values(["familia", METRICA], ascending=[True, False]), extra=("familia",))

### Costo y épocas de cada fold

`segundos` es el tiempo de entrenamiento de cada fold que guardó el 16 (en las CNN suma las dos etapas). Las
mejores épocas son las que eligió el val oficial no jamaicano.

In [ ]:
filas = []
for corrida in CON_CV:
    modelo, modo = corrida.split("/")
    for fold in folds:
        archivo = cv_jamaica.ruta_fold(rutas, modelo, modo, fold["fold"]) / "entrenamiento.json"
        registro = json.loads(archivo.read_text(encoding="utf-8"))
        filas.append({"corrida": corrida, "fold": fold["fold"], "segundos": registro["segundos"],
                      **{f"mejor_epoca_{etapa}": datos["mejor_epoca"] for etapa, datos in registro["etapas"].items()}})
costos = pd.DataFrame(filas)
mediana = costos.drop(columns="fold").groupby("corrida").median()
display(costos.set_index(["corrida", "fold"]))
mediana.round(1)

### Umbral

`umbral_referencia` es el actual. `umbral` es el que se elegiría con todas las imágenes de Jamaica, y
`diferencia_cv` mide cuánto cambia la balanced accuracy si el umbral se elige dejando afuera un grupo espacial
por vez. Solo `mejora_clara = True` justifica cambiarlo.

In [ ]:
umbrales = cv_jamaica.tabla_umbrales(cv)
umbrales["corrida"] = umbrales["modelo"] + "/" + umbrales["modo"]
umbrales = umbrales.set_index("corrida")
umbrales[["umbral_referencia", "umbral", "umbral_cv_min", "umbral_cv_max", "balanced_accuracy_referencia",
          "balanced_accuracy_cv", "diferencia_cv", "diferencia_cv_inf", "diferencia_cv_sup", "mejora_clara"]].round(3)

## 3. Decisión

Se aplica la regla tal cual, familia por familia.

In [ ]:
def intervalo(fila, metrica):
    return f"{fila[metrica]:.3f} [{fila[metrica + '_inf']:.3f}, {fila[metrica + '_sup']:.3f}]"


decisiones = []
for familia, nombre in FAMILIAS.items():
    finalistas = [c for c in FINALISTAS if familia_de(c) == familia]
    candidatos = [c for c in finalistas if c in CON_CV]
    sin_cv = [c for c in finalistas if c not in CON_CV and c not in OPCIONALES]
    if not candidatos:
        decisiones.append({"familia": nombre, "modelo final": "—", "motivo": "ningún finalista tiene los cinco folds"})
        continue
    orden = (resumen_cv[resumen_cv["corrida"].isin(candidatos)]
             .sort_values(METRICA, ascending=False)["corrida"].tolist())
    elegido = orden[0]
    if len(orden) == 1:
        motivo = "único finalista con los cinco folds"
    else:
        d = comparacion.diferencias(cv, orden[:2], referencia=orden[1], metrica=METRICA).iloc[0]
        texto = f"{orden[0]} − {orden[1]} = {d['diferencia']:+.3f} [{d['inf']:+.3f}, {d['sup']:+.3f}]"
        if d["clara"]:
            motivo = f"mayor balanced accuracy en la CV, diferencia clara ({texto})"
        else:
            elegido = min(orden[:2], key=lambda c: mediana.loc[c, "segundos"])
            motivo = f"diferencia no clara ({texto}): gana el de menor costo de entrenamiento"
    if sin_cv:
        motivo += f"; sin rival en la CV (falta completar {', '.join(sin_cv)})"
    fila_umbral = umbrales.loc[elegido]
    umbral_final = fila_umbral["umbral"] if fila_umbral["mejora_clara"] else fila_umbral["umbral_referencia"]
    fila_cv = resumen_cv.set_index("corrida").loc[elegido]
    fila_val = val.set_index("corrida").loc[elegido] if elegido in VIGENTES else None
    epocas = mediana.loc[elegido].drop("segundos").dropna()
    decisiones.append({
        "familia": nombre,
        "modelo final": elegido,
        "motivo": motivo,
        "umbral": float(umbral_final),
        "umbral cambia": bool(fila_umbral["mejora_clara"]),
        "balanced accuracy CV": intervalo(fila_cv, METRICA),
        "F1 macro val": "—" if fila_val is None else intervalo(fila_val, "f1_macro"),
        "épocas (mediana de folds)": ", ".join(f"{k.removeprefix('mejor_epoca_')}: {v:g}" for k, v in epocas.items()),
    })

decision = pd.DataFrame(decisiones).set_index("familia")
with pd.option_context("display.max_colwidth", None):
    display(decision.T)

### Clasificador contra detector, con los modelos elegidos

La pregunta de la consigna, sobre las predicciones fuera de fold de Jamaica: diferencia pareada *detector −
CNN*. El **AUC** no depende del umbral. Las otras métricas usan el umbral de cada uno.

In [ ]:
elegidos = decision["modelo final"].tolist()
if len(elegidos) == 2 and "—" not in elegidos:
    cnn, detector = elegidos
    filas = []
    for metrica in ("roc_auc", "balanced_accuracy", "f1_macro", "recall_sin_paneles"):
        d = comparacion.diferencias(cv, [detector], cnn, metrica=metrica).iloc[0]
        filas.append({"métrica": TITULOS[metrica], "detector − CNN": d["diferencia"], "inf": d["inf"],
                      "sup": d["sup"], "clara": d["clara"]})
    display(pd.DataFrame(filas).set_index("métrica").round(3))
else:
    print("Falta un modelo elegido en alguna familia: no se comparan.")

## 4. Después de correr

1. Copiar la tabla de la sección 3 en [`docs/modelo_final.md`](../docs/modelo_final.md), con la fecha y el
   commit. El 18 toma de ahí los modelos finales.
2. Correr `18_evaluacion_test` **una sola vez**, una sesión por familia. No volver a este notebook para cambiar
   la elección después de ver el test.
3. En el informe:
   - decir que val eligió los finalistas y los checkpoints, y que la CV de Jamaica decidió entre ellos;
   - si una familia quedó *sin rival en la CV*, decirlo y explicar qué candidato faltó;
   - aclarar que hubo una corrida preliminar del test el 6/11 y que la excepción de la regla es posterior;
   - aclarar que los modelos se entrenaron solo con train (no hubo reentrenamiento con train + val).